In [1]:
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

# ==============================
# 1. CẤU HÌNH & TIỀN XỬ LÝ
# ==============================
TEENCODE = {"ko": "không", "k": "không", "hok": "không", "dc": "được", "sv": "sinh viên", "mk": "mình"}

def clean_text(text):
    if pd.isna(text) or text == "": return ""
    text = text.lower()
    words = [TEENCODE.get(w, w) for w in text.split()]
    text = re.sub(r"[^a-zA-Zà-ỹ\s]", " ", " ".join(words))
    return re.sub(r"\s+", " ", text).strip()

# ==============================
# 2. LOAD DATA
# ==============================
# Thay đổi đường dẫn file của bạn tại đây
train_df = pd.read_csv('train.csv') 

# Clean văn bản ngay từ đầu
train_df["Personal_Essay_clean"] = train_df["Personal_Essay"].apply(clean_text)
train_df["Advisor_Notes_clean"] = train_df["Advisor_Notes"].apply(clean_text)

# ==============================
# 3. FEATURE ENGINEERING
# ==============================
def extract_features(df):
    # Text features
    df["essay_word_count"] = df["Personal_Essay_clean"].apply(lambda x: len(x.split()))
    df["advisor_note_words"] = df["Advisor_Notes_clean"].apply(lambda x: len(x.split()))
    
    # Attendance features (giả sử có các cột Attendance_1, Attendance_2...)
    attn_cols = [c for c in df.columns if "Attendance" in c]
    if attn_cols:
        df["attendance_mean"] = df[attn_cols].replace(-1, np.nan).mean(axis=1).fillna(0)
    else:
        df["attendance_mean"] = 0
        
    return df

train_df = extract_features(train_df)

# ==============================
# 4. VECTORIZATION (TF-IDF)
# ==============================
tfidf = TfidfVectorizer(max_features=200, ngram_range=(1,2), min_df=3)
tfidf_matrix = tfidf.fit_transform(train_df["Personal_Essay_clean"])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

# Kết hợp features
X = pd.concat([
    train_df[["essay_word_count", "advisor_note_words", "attendance_mean"]].reset_index(drop=True),
    tfidf_df
], axis=1)

y = train_df["Academic_Status"]

# ==============================
# 5. TRAIN MODEL
# ==============================
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    use_label_encoder=False
)
model.fit(X_train, y_train)

print(f"Validation Accuracy: {model.score(X_val, y_val):.4f}")

# ==============================
# 6. SERIALIZE (QUAN TRỌNG NHẤT)
# ==============================
# Lưu mọi thứ vào 1 file pkl duy nhất
artifacts = {
    "model": model,
    "tfidf": tfidf,
    "feature_names": X.columns.tolist(),
    "teencode": TEENCODE
}

with open("model_assets.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Đã lưu file model_assets.pkl thành công!")

c:\Users\Nguyen Hoang Long\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [10:54:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation Accuracy: 0.7292
Đã lưu file model_assets.pkl thành công!
